# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the `mlcroissant` library, referencing all Croissant schema entities by their `@id`.

### Dataset Source
The dataset is defined by a Croissant schema and is publicly available at the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. Metadata fields are accessed directly as attributes.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets (tables), their fields, and get their `@id` references.

**Tip:** All `mlcroissant` references should use the Croissant `@id` fields, not just names or positions.

In [ ]:
# List all record sets (tables) and their IDs
record_set_entities = dataset.metadata.record_sets
print("Available Record Sets:")
for rs in record_set_entities:
    print(f"  - name: {rs.name}, @id: {rs.id}")

# Explore fields (columns) for each record set
for rs in record_set_entities:
    field_ids = [f.id for f in rs.fields]
    print(f"\nRecord Set '{rs.name}' (@id: {rs.id}) fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}), type: {getattr(field, 'data_type', None)}")

## 3. Data Extraction
Load data from each record set into a DataFrame, referencing by record set `@id`.

Let's extract the data for the available record set(s), and inspect one of them.

In [ ]:
# Gather all record set @ids
record_sets = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded '{record_set_id}': shape {df.shape}")

# Show columns for the first record set
if record_sets:
    first_rs_id = record_sets[0]
    print(f"Columns for record set {first_rs_id}:")
    print(dataframes[first_rs_id].columns.tolist())
    dataframes[first_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
We'll process fields by referencing their Croissant `@id`.

For this section, we select a numeric field and a grouping (categorical) field, by their `@id`, and perform outlier removal, normalization, and grouping operations.

*(Note: replace the variables below with the actual `@id` for fields as per section 2 above. For demonstration, we will select two plausible field IDs from the field list.)*

In [ ]:
# Example: Suppose we have a field for 'Interval_between_diagnoses' and 'Sex' (edit as needed)
record_set_id = record_sets[0]  # Use the first (or principal) record set
df = dataframes[record_set_id]

# Choose the actual field @id for a numeric field (e.g., interval between primary and secondary cancer in months)
# and a categorical field (e.g., sex or anatomical site). Find the proper @id from previous listing.
# Replace these with real '@id' values as shown in section 2 output.
numeric_field_id = None
group_field_id = None

# Try to autodetect a numeric field by scanning the columns for common keywords
import re
for col in df.columns:
    if re.search(r'interv|month|age|metasta|stage', col, re.IGNORECASE):
        numeric_field_id = col
        break
for col in df.columns:
    if re.search(r'sex|gender|site|location', col, re.IGNORECASE):
        group_field_id = col
        break
print(f"Numeric field: {numeric_field_id}")
print(f"Group field: {group_field_id}")

# Proceed with analysis if the numeric field exists
if numeric_field_id is not None and pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    threshold = df[numeric_field_id].quantile(0.10)  # Filter outliers below 10th percentile
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (10th percentile):")
    print(filtered_df[[numeric_field_id]].head())
    
    # Normalize numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].copy()].head())
    
    # Group by group_field_id and show means
    if group_field_id is not None and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No suitable numeric field detected for demonstration EDA.")

## 5. Visualization
Visualize distributions or groupings of key fields (referenced by `@id`).

We'll plot the distribution of the numeric field and compare it across group categories.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id], bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    
    if group_field_id is not None and group_field_id in df.columns:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} grouped by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No numeric field found for visualization.")

## 6. Conclusion
In this notebook, we've demonstrated loading and exploring the FAIR^2 dataset using the `mlcroissant` library, referencing all data entities by `@id` as recommended by the Croissant specification.

- We loaded metadata and records directly from the Croissant schema URL.
- We inspected record sets and fields, referencing all by their `@id`.
- We demonstrated typical data processing: filtering, normalization, grouping, and visualizations, always referencing fields by their Croissant `@id`.

**Next Steps:** Use the detailed field and group IDs as shown by the code completion above to tailor more advanced analyses, including statistical modeling, subcohort analyses, or exporting data for downstream ML tasks.